# Monte-Carlo Payoffs

Author: Sebastien Gurrieri, sebgur@gmail.com

This notebook illustrates the pricing of a number of payoffs by Monte-Carlo, using the payoff library.

In [7]:
import datetime as dt
import numpy as np
from sdevpy.utilities.book import Book
# from scipy.stats import norm
# import matplotlib.pyplot as plt
import sdevpy as sd
from sdevpy.market import provider as mdp
from sdevpy.montecarlo.payoffs import cashflows as cfl
from sdevpy.montecarlo.payoffs.basic import Trade, Instrument, Variance
from sdevpy.montecarlo.payoffs.vanillas import make_vanilla_option
from sdevpy.montecarlo.payoffs.exotics import WorstOfBarrier, make_basket_option, make_asian_option
from sdevpy.montecarlo import mcpricer as mc
# from sdevpy.maths.metrics import rmse
from sdevpy.analytics import black
# from sdevpy.volatility.localvol import localvol_factory as lvf
# from sdevpy.utilities import timegrids
# from sdevpy.utilities.timegrids import TimeGridBucket
# from sdevpy.utilities import dates as dts

print("SDevPy version: " + sd.__version__)

SDevPy version: 1.1


### Setup Date and MarketDataProvider
We assume here that all market data is available and all required models have been calibrated.

In [8]:
valdate = dt.datetime(2025, 12, 15)
md = mdp.MarketDataFileProvider()

### Setup Portfolio
In this section we create a few trades, showing example of various ways instruments can be created. The vanilla ones will later be used for comparison against closed-forms.

In [9]:
# Create portfolio
book = Book()
trades = []
start_date = dt.datetime(2025, 11, 15)
expiry = dt.datetime(2026, 12, 15)
expiry2 = dt.datetime(2027, 12, 15)
v_name, v_strike, v_type = 'ABC', 100.0, 'Call' # For check against CF

# Vanilla
index = make_vanilla_option(v_name, v_strike, v_type, expiry)
cf = cfl.Cashflow(index, expiry)
trades.append(Trade(Instrument(cashflow_legs=[[cf]])))

# Basket option
index = make_basket_option(['XYZ', 'KLM'], [0.5, 0.1], 100.0, 'Call', expiry)
cf = cfl.Cashflow(index, expiry)
trades.append(Trade(Instrument(cashflow_legs=[[cf]])))

# Asian option
index = make_asian_option('ABC', 100.0, 'Call', valdate, expiry, freq='5D')
cf = cfl.Cashflow(index, expiry)
trades.append(Trade(Instrument(cashflow_legs=[[cf]])))

# Worst-of barrier
index = WorstOfBarrier(['ABC', 'XYZ'], expiry, 100.0, 'Call', 35.0)
cf = cfl.Cashflow(index, expiry)
trades.append(Trade(Instrument(cashflow_legs=[[cf]])))

# VarSwap
index = Variance('ABC', start_date, expiry)
vstrike = 14.0
payoff = index - vstrike * vstrike
cf = cfl.Cashflow(payoff, expiry)
trades.append(Trade(Instrument(cashflow_legs=[[cf]])))

# Create book
book.add_trades(trades)

### Launch the MC simulation
The simulation is triggered with a number of optional parameters related to the path construction.

In [10]:
# Path parameters
constr_type = 'brownianbridge' # brownianbridge, incremental
rng_type = 'sobol' # sobol, mt, halton, latinhypercube
n_paths = 1 * 1000
n_steps = 50

# Price book
mc_price = mc.price_book(valdate, book, md, constr_type=constr_type, rng_type=rng_type,
                         n_paths=n_paths, n_timesteps=n_steps)

# Display prices

TypeError: 'NoneType' object is not iterable

In [ ]:
# Gather all names in the book
names = book.names
print(f"Book names: {names}")
print(f"Number of assets: {len(names)}")

# Closed-form for vanilla
md = mdp.MarketDataFileProvider()
disc_curve = md.get_yieldcurve(book.csa_curve_id, valdate)
fwd_curves = mdp.get_eq_forward_curves(names, valdate, md)
lvs = get_local_vols(names, valdate)
name_idx = names.index(v_name)
fwd = fwd_curves[name_idx].value(expiry)
df = disc_curve.discount(expiry)
ttm = timegrids.model_time(valdate, expiry)
sigma = np.asarray([0.2] * len(names))
cf_price = df * black.price(ttm, v_strike, v_type, fwd, sigma[name_idx])

for i in range(len(mc_price['pv'])):
    print("MC:", mc_price['pv'][i])

print("CF:", cf_price)